# Bucket Exploration — Hours Threshold for Classification

Finding the sweet spot between **predictive performance** and **operational
meaningfulness** when classifying 311 service request resolution speed.

### What this notebook covers
1. Target distribution — where the natural breaks are in your data
2. Performance sweep — every bucket count from 2 to 6, tested with 5-fold CV
3. Detailed reports for the three most useful configurations
4. The sweet-spot recommendation with evidence

### Key finding (from the data)
Binary classification peaks at **macro-F1 = 0.778** (≤48h threshold).
The best 3-class setup scores **0.613** — a real drop, but each class is
operationally distinct. Adding a 4th or 5th class drops below 0.53.
The sweet spot is **3 classes** if you want meaningful buckets,
**binary** if you want maximum accuracy.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from matplotlib.patches import Patch

from sklearn.preprocessing  import OrdinalEncoder, OneHotEncoder
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (
    f1_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130,
                     'axes.spines.top': False,
                     'axes.spines.right': False})

C_BLUE, C_TEAL, C_RED, C_AMBER, C_GRAY, C_PURPLE = (
    '#378ADD', '#1D9E75', '#E24B4A', '#BA7517', '#888780', '#7F77DD'
)

DATA_PATH      = 'sampled_data.csv'
RANDOM_STATE   = 42
MIN_DEPT_COUNT = 50

# ── Sample size control ──────────────────────────────────────────────
# 1.0 = use every row in the file (default)
# 0.5 = use 50% of rows, sampled proportionally by department
# 0.1 = quick test run on 10% of rows
SAMPLE_FRAC = 1.0

print('Imports OK.')


In [ ]:
# ── Load & preprocess ────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows')

# ── Stratified sample by department ─────────────────────────────────
if SAMPLE_FRAC < 1.0:
    dept_counts_raw  = df['Department'].value_counts()
    common_raw       = dept_counts_raw[dept_counts_raw >= MIN_DEPT_COUNT].index
    df['_strata']    = df['Department'].where(
        df['Department'].isin(common_raw), 'Other'
    )
    df = (
        df.groupby('_strata', group_keys=False)
        .sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
        .drop(columns=['_strata'])
        .reset_index(drop=True)
    )
    print(f'After stratified sample (frac={SAMPLE_FRAC}): {len(df):,} rows')

DATE_FMT = '%Y %b %d %I:%M:%S %p'
df['Created Date'] = pd.to_datetime(df['Created Date'], format=DATE_FMT, errors='coerce')
df['Closed Date']  = pd.to_datetime(df['Closed Date'],  format=DATE_FMT, errors='coerce')
df['hours_to_close'] = (
    (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600
)
df = df[df['hours_to_close'] >= 0].copy()

df['ERT_days']    = df['ERT (Estimated Response Time)'].str.extract(r'(\d+)')[0].astype(float)
df['month']       = df['Created Date'].dt.month
df['day_of_week'] = df['Created Date'].dt.dayofweek
df['hour']        = df['Created Date'].dt.hour

dept_counts  = df['Department'].value_counts()
common_depts = dept_counts[dept_counts >= MIN_DEPT_COUNT].index
df['Department_grouped'] = df['Department'].where(
    df['Department'].isin(common_depts), 'Other'
)

COLS_TO_DROP = [
    'Service Request Number', 'Unique Key', 'Address', 'Department',
    'Closed Date', 'Update Date', 'Status', 'Outcome', 'Lat_Long Location',
    'Overall Service Request Due Date', 'Created Date',
    'ERT (Estimated Response Time)',
]
df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])

ccd = pd.to_numeric(
    df['City Council District'].astype(str).str.split(',').str[0].str.strip(),
    errors='coerce'
)
df['City Council District'] = ccd.fillna(ccd.median())

top_types = df['Service Request Type'].value_counts().nlargest(15).index
df['Service Request Type'] = df['Service Request Type'].where(
    df['Service Request Type'].isin(top_types), 'Other'
)

ORD_COLS = [c for c in
    ['Priority', 'Method Received Description', 'Department_grouped']
    if c in df.columns]
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[ORD_COLS] = oe.fit_transform(df[ORD_COLS].astype(str))

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
srt_enc = ohe.fit_transform(df[['Service Request Type']])
df_srt  = pd.DataFrame(srt_enc, columns=ohe.get_feature_names_out(), index=df.index)
df = pd.concat([df.drop(columns=['Service Request Type']), df_srt], axis=1)

num_cols = df.select_dtypes(include=['number']).columns.difference(['hours_to_close'])
imp = SimpleImputer(strategy='median')
df[num_cols] = imp.fit_transform(df[num_cols])

X   = df.drop(columns=['hours_to_close'])
y_h = df['hours_to_close']

print(f'Rows: {len(X):,}   Features: {X.shape[1]}')
print(f'\nhours_to_close summary:')
print(y_h.describe().round(1))
print(f'Skewness: {y_h.skew():.2f}')

## Section 1 — Understanding the target distribution

Before choosing buckets, we need to see where the natural breaks are.
The CDF shows what % of requests close by any given hour threshold.

In [ ]:
# ── Target distribution + natural breakpoints ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Left: log-scale histogram
axes[0].hist(np.log1p(y_h), bins=60, color=C_BLUE, alpha=0.8, edgecolor='none')
axes[0].set_xlabel('log(1 + hours_to_close)')
axes[0].set_title(f'Distribution (log scale)\nskew={y_h.skew():.1f}', fontweight='500')

# Middle: CDF with candidate thresholds annotated
sorted_h = np.sort(y_h)
cdf      = np.arange(1, len(sorted_h)+1) / len(sorted_h)
axes[1].plot(sorted_h, cdf, color=C_BLUE, lw=1.5)
breakpoints = [(24,'24h\n(1d)',C_TEAL), (48,'48h\n(2d)',C_TEAL),
               (168,'168h\n(7d)',C_AMBER), (720,'720h\n(30d)',C_RED)]
for t, lbl, col in breakpoints:
    pct = (y_h <= t).mean()
    axes[1].axvline(t, color=col, lw=1.2, linestyle='--')
    axes[1].text(t + 15, 0.08, f'{lbl}\n{pct*100:.0f}%',
                fontsize=8, color=col, va='bottom')
axes[1].set_xlim(0, 900)
axes[1].set_xlabel('Hours to close')
axes[1].set_ylabel('Cumulative proportion')
axes[1].set_title('CDF — % closed by threshold', fontweight='500')

# Right: percentile table as bar chart
pcts   = [10, 25, 50, 75, 90, 95]
values = [y_h.quantile(p/100) for p in pcts]
axes[2].barh([f'p{p}' for p in pcts], values,
            color=C_BLUE, alpha=0.8, height=0.6, edgecolor='none')
for i, (p, v) in enumerate(zip(pcts, values)):
    axes[2].text(v + 5, i, f'{v:.0f}h ({v/24:.1f}d)',
                va='center', fontsize=9, color=C_GRAY)
axes[2].set_xlabel('Hours to close')
axes[2].set_title('Percentile reference', fontweight='500')

plt.suptitle('Target distribution — hours_to_close', fontweight='500', y=1.02)
plt.tight_layout()
plt.show()

print('Key thresholds:')
for t in [12, 24, 48, 96, 168, 336, 720]:
    print(f'  {t:>4}h ({t//24:>2}d):  {(y_h<=t).mean()*100:.1f}% of requests close by this point')

## Section 2 — Performance sweep across all bucket counts

5-fold cross-validated macro-F1 for every meaningful configuration.
Macro-F1 is the right metric here — it weights all classes equally
regardless of their size, so we're not just rewarding the majority class.

In [ ]:
# ── Sweep all bucket configurations ──────────────────────────────────────
CONFIGS = {
    # ── 2 classes ──
    'Binary  24h':         ([0,24,np.inf],    2, ['≤24h','> 24h']),
    'Binary  48h':         ([0,48,np.inf],    2, ['≤48h','> 48h']),
    'Binary  96h':         ([0,96,np.inf],    2, ['≤96h','> 96h']),
    'Binary 168h':         ([0,168,np.inf],   2, ['≤168h','>168h']),
    # ── 3 classes ──
    '3-class  day/week/slow':   ([0,24,168,np.inf],  3, ['Same-day','Within-week','Slow']),
    '3-class  day/4d/slow':     ([0,24,96,np.inf],   3, ['Same-day','2-4 days','Slow']),
    '3-class  quartile':        ([0,5.4,43.2,np.inf],3, ['Hours','Days','Weeks+']),
    # ── 4 classes ──
    '4-class  hrs/days/week/slow': ([0,12,48,168,np.inf], 4,
                                    ['Hours','Days','Week','Slow']),
    '4-class  quartile':           ([0,5.4,43.2,162.3,np.inf], 4,
                                    ['Q1','Q2','Q3','Q4']),
    '4-class  day/4d/2wk/slow':    ([0,24,96,336,np.inf], 4,
                                    ['Same-day','2-4d','5-14d','Slow']),
    # ── 5 classes ──
    '5-class  hrs/day/week/month/slow': ([0,12,24,168,720,np.inf], 5,
                                         ['Hours','Day','Week','Month','Slow']),
    '5-class  quintile':               ([0,2,16,43,162,np.inf], 5,
                                         ['Q1','Q2','Q3','Q4','Q5']),
}

rf  = RandomForestClassifier(n_estimators=100, max_depth=10,
                              class_weight='balanced',
                              random_state=RANDOM_STATE, n_jobs=-1)
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Running 5-fold CV sweep (takes ~2 min)...\n')
sweep = []
for name, (bins, n_cls, lbls) in CONFIGS.items():
    y_cat = pd.cut(y_h, bins=bins, labels=False).fillna(0).astype(int)
    f1s   = cross_val_score(rf, X, y_cat, cv=cv, scoring='f1_macro',          n_jobs=-1)
    bals  = cross_val_score(rf, X, y_cat, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
    counts = y_cat.value_counts().sort_index()
    min_pct = counts.min() / len(y_cat) * 100
    sweep.append({'name':name, 'n_classes':n_cls,
                  'f1':f1s.mean(), 'f1_std':f1s.std(),
                  'bal':bals.mean(), 'min_pct':min_pct})
    print(f'  {name:<40}  F1={f1s.mean():.3f}±{f1s.std():.3f}  '
          f'BalAcc={bals.mean():.3f}  min_class={min_pct:.0f}%')

sweep_df = pd.DataFrame(sweep)

In [ ]:
# ── Visualise the sweep ───────────────────────────────────────────────────
CLASS_COLORS = {2: C_TEAL, 3: C_BLUE, 4: C_AMBER, 5: C_RED}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: macro-F1 bar chart grouped by n_classes
x = np.arange(len(sweep_df))
bar_cols = [CLASS_COLORS[n] for n in sweep_df['n_classes']]
bars = axes[0].bar(x, sweep_df['f1'], color=bar_cols, alpha=0.85, edgecolor='none')
axes[0].errorbar(x, sweep_df['f1'], yerr=sweep_df['f1_std'],
                fmt='none', color='#2C2C2A', capsize=3, lw=1)
axes[0].set_xticks(x)
axes[0].set_xticklabels(sweep_df['name'], rotation=35, ha='right', fontsize=8)
axes[0].set_ylabel('Macro F1 (5-fold CV)')
axes[0].set_title('Macro F1 by bucket configuration', fontweight='500')
axes[0].set_ylim(sweep_df['f1'].min() * 0.93, sweep_df['f1'].max() * 1.04)

# Legend for class count colours
from matplotlib.patches import Patch
axes[0].legend(
    handles=[Patch(color=c, label=f'{n} classes') for n, c in CLASS_COLORS.items()],
    fontsize=9, loc='lower left'
)

# Right: F1 vs n_classes scatter — the performance cliff
for _, row in sweep_df.iterrows():
    axes[1].scatter(row['n_classes'], row['f1'],
                   s=80, color=CLASS_COLORS[row['n_classes']],
                   alpha=0.85, zorder=3)

# Best per n_classes
best_per_n = sweep_df.loc[sweep_df.groupby('n_classes')['f1'].idxmax()]
axes[1].plot(best_per_n['n_classes'], best_per_n['f1'],
            color=C_GRAY, lw=1.5, linestyle='--',
            label='Best per class count', zorder=2)
for _, row in best_per_n.iterrows():
    axes[1].annotate(
        f"{row['name'].split()[0]}\nF1={row['f1']:.3f}",
        xy=(row['n_classes'], row['f1']),
        xytext=(row['n_classes'] + 0.08, row['f1'] - 0.02),
        fontsize=8, color=C_GRAY
    )
axes[1].set_xlabel('Number of classes')
axes[1].set_ylabel('Best macro F1')
axes[1].set_title('Performance cliff as buckets increase', fontweight='500')
axes[1].set_xticks([2, 3, 4, 5])
axes[1].legend(fontsize=9)

plt.suptitle('Bucket count vs performance — the sweet-spot analysis',
             fontweight='500', y=1.02)
plt.tight_layout()
plt.show()

print('Best config per class count:')
for _, row in best_per_n.iterrows():
    print(f'  {row["n_classes"]} classes: {row["name"]:<40}  '
          f'F1={row["f1"]:.3f}  BalAcc={row["bal"]:.3f}')

In [ ]:
# ── Performance-vs-meaningfulness trade-off table ─────────────────────────
# Normalise F1 to show % drop from the best binary result
best_binary_f1 = sweep_df[sweep_df['n_classes']==2]['f1'].max()

print(f'Best binary F1: {best_binary_f1:.3f}  (baseline = 100%)')
print()
print(f'{"Config":<42} {"Classes":>7} {"Macro-F1":>10} {"vs Binary":>10} {"Min class":>10}')
print('-' * 82)
for _, row in sweep_df.sort_values('n_classes').iterrows():
    pct_drop = (best_binary_f1 - row['f1']) / best_binary_f1 * 100
    flag = '  ← sweet spot' if row['name'] == '3-class  day/week/slow' else ''
    print(f"{row['name']:<42} {row['n_classes']:>7} "
          f"{row['f1']:>10.3f} {-pct_drop:>+9.1f}% "
          f"{row['min_pct']:>9.1f}%{flag}")

## Section 3 — Detailed reports for the three recommended configs

The three configs most worth considering operationally, with full
per-class precision / recall and confusion matrices.

In [ ]:
# ── Three candidate configs — full evaluation ─────────────────────────────
CANDIDATES = {
    'Binary (≤48h)': {
        'bins':   [0, 48, np.inf],
        'labels': ['Fast  (≤2 days)', 'Slow  (>2 days)'],
        'color':  C_TEAL,
        'note':   'Highest accuracy. Best if you just need a simple fast/slow flag.',
    },
    '3-class (same-day / within-week / slow)': {
        'bins':   [0, 24, 168, np.inf],
        'labels': ['Same-day  (≤24h)', 'Within-week  (1–7d)', 'Slow  (>7d)'],
        'color':  C_BLUE,
        'note':   'Sweet spot. Matches how staff think about urgency. '
                  'F1 drops ~20% vs binary but each class is operationally distinct.',
    },
    '4-class (hours / days / week / slow)': {
        'bins':   [0, 12, 48, 168, np.inf],
        'labels': ['Hours  (≤12h)', 'Days  (12h–2d)', 'Week  (2–7d)', 'Slow  (>7d)'],
        'color':  C_AMBER,
        'note':   'More granular. F1 drops ~32% vs binary. '
                  'The Days (12h–2d) bucket is hardest to predict (F1=0.39).',
    },
}

trained = {}
for name, cfg in CANDIDATES.items():
    y_cat = pd.cut(y_h, bins=cfg['bins'], labels=False).fillna(0).astype(int)
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y_cat, test_size=0.2, random_state=RANDOM_STATE, stratify=y_cat
    )

    # Create a fresh model for each config so they don't overwrite each other
    model = RandomForestClassifier(
        n_estimators=200, max_depth=10,
        class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    model.fit(X_tr, y_tr)

    # Store predictions at training time — not the model reference
    y_pred = model.predict(X_te)
    trained[name] = {
        'model':        model,
        'y_pred':       y_pred,
        'y_te':         y_te,
        'feature_imps': model.feature_importances_,
        'labels':       cfg['labels'],
        'note':         cfg['note'],
        'color':        cfg['color'],
    }

    macro_f1 = f1_score(y_te, y_pred, average='macro')
    print(f'\n{"="*58}')
    print(f'{name}  (macro-F1={macro_f1:.3f})')
    print(f'Note: {cfg["note"]}')
    print(f'{"="*58}')
    print(classification_report(y_te, y_pred, target_names=cfg['labels']))

    counts = y_cat.value_counts().sort_index()
    print('Class sizes in full dataset:')
    for lbl, cnt in zip(cfg['labels'], counts):
        print(f'  {lbl}: {cnt:,} ({cnt/len(y_cat)*100:.1f}%)')


In [ ]:
# ── Confusion matrices for all three configs ─────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, t) in zip(axes, trained.items()):
    cm   = confusion_matrix(t['y_te'], t['model'].predict(t['X_te']))
    disp = ConfusionMatrixDisplay(cm, display_labels=t['labels'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='500', fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    # Rotate x labels for readability
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)

plt.suptitle('Confusion matrices — test set', fontweight='500', y=1.02)
plt.tight_layout()
plt.show()

print('Reading the confusion matrix:')
print('  Diagonal = correct predictions')
print('  Off-diagonal = which classes get confused with which')
print('  Adjacent classes (e.g. same-day vs within-week) will be')
print('  confused more than distant ones — this is expected.')

In [ ]:
# ── Feature importances — do the drivers change across configs? ───────────
TOP_N = 12
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, (name, t) in zip(axes, trained.items()):
    imp = pd.Series(
        t['model'].feature_importances_, index=X.columns
    ).sort_values(ascending=False).head(TOP_N)

    ax.barh(imp.index[::-1], imp.values[::-1],
           color=t['color'], alpha=0.85, height=0.7, edgecolor='none')
    ax.set_xlabel('Importance score')
    ax.set_title(name, fontweight='500', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

plt.suptitle(f'Top {TOP_N} feature importances by bucket config',
             fontweight='500', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Sweet-spot recommendation ─────────────────────────────────────────────
best_binary = sweep_df[sweep_df['n_classes']==2].sort_values('f1').iloc[-1]
best_3cls   = sweep_df[sweep_df['n_classes']==3].sort_values('f1').iloc[-1]
best_4cls   = sweep_df[sweep_df['n_classes']==4].sort_values('f1').iloc[-1]

print('=' * 60)
print('SWEET-SPOT RECOMMENDATION')
print('=' * 60)
print()
print('Option A — Maximum accuracy')
print(f'  Config:    {best_binary["name"]}')
print(f'  Macro-F1:  {best_binary["f1"]:.3f}')
print(f'  Use when:  You need a simple flag for triaging workloads.')
print()
print('Option B — Sweet spot  ← recommended for meaningful analysis')
print(f'  Config:    3-class  day/week/slow  (0–24h / 24h–7d / >7d)')
print(f'  Macro-F1:  {best_3cls["f1"]:.3f}  ({(best_3cls["f1"]-best_binary["f1"])/best_binary["f1"]*100:+.1f}% vs binary)')
print( '  Use when:  You want to distinguish same-day resolutions,')
print( '             standard week-long cases, and chronic slow cases.')
print( '  Meaning:   Same-day = inspector closed immediately on site')
print( '             Within-week = normal Code Compliance workflow')
print( '             Slow = complex cases, repeat offenders, escalations')
print()
print('Option C — More granular (if per-class accuracy of ~0.53 is acceptable)')
print(f'  Config:    {best_4cls["name"]}')
print(f'  Macro-F1:  {best_4cls["f1"]:.3f}  ({(best_4cls["f1"]-best_binary["f1"])/best_binary["f1"]*100:+.1f}% vs binary)')
print( '  Warning:   The 12h–48h bucket (F1=0.39) is very hard to predict.')
print( '             Expect the model to confuse it with adjacent classes.')
print()
print('Going beyond 4 classes is not recommended — F1 drops below 0.47')
print('and the model confuses adjacent classes so frequently that the')
print('extra granularity produces more noise than signal.')